# Customer Churn Prediction — Step 3: Data Cleaning

**Dataset:** Telco Customer Churn (Kaggle / IBM Watson Analytics)  
**Input:** `data/WA_Fn-UseC_-Telco-Customer-Churn.csv` (raw, never modified)  
**Output:** `data/telco_churn_cleaned.csv` (new file — raw CSV is untouched)

### What this notebook does
Applies every fix identified in the Step 2 Data Quality Audit to produce a clean DataFrame (`df_clean`) that is ready for Exploratory Data Analysis.  

### What this notebook does NOT do
- Does **not** overwrite or modify the raw CSV.
- Does **not** perform feature encoding or scaling.
- Does **not** perform train/test splitting.
- Does **not** train any machine learning model.
- Every decision is explained before the code that implements it.

---

## 1. Setup — Import Libraries and Load Raw Data

We load the same libraries as in Steps 1 and 2.  
`pandas` does all the heavy lifting; `numpy` is needed for `np.inf` / `np.nan` sentinel checks.  

The raw DataFrame `df` is loaded once and is **never written to** in this notebook.  
All cleaning happens on a copy called `df_clean`.

In [67]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 120)

# ── Resolve paths ────────────────────────────────────────────────────────────
NOTEBOOK_DIR  = os.path.abspath('')
PROJECT_ROOT  = os.path.dirname(NOTEBOOK_DIR) if os.path.basename(NOTEBOOK_DIR) == 'notebook' else NOTEBOOK_DIR
RAW_DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
CLEAN_PATH    = os.path.join(PROJECT_ROOT, 'data', 'telco_churn_cleaned.csv')

print('Raw data  :', RAW_DATA_PATH)
print('Clean out :', CLEAN_PATH)
print('Raw file exists:', os.path.exists(RAW_DATA_PATH))

Raw data  : c:\Users\Subham\OneDrive\Documents\Desktop\Customer_Churn_Project\data\WA_Fn-UseC_-Telco-Customer-Churn.csv
Clean out : c:\Users\Subham\OneDrive\Documents\Desktop\Customer_Churn_Project\data\telco_churn_cleaned.csv
Raw file exists: True


In [68]:
# Load the raw CSV — no coercions, exactly as on disk
df = pd.read_csv(RAW_DATA_PATH)

print(f'Raw DataFrame loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

Raw DataFrame loaded: 7,043 rows × 21 columns


---
## 2. Baseline Snapshot (Before Cleaning)

Before touching anything, we record the key statistics of the raw DataFrame.  
These numbers will be compared against the cleaned DataFrame at the end of the notebook to produce the **before-vs-after summary**.

Recording the baseline now also ensures that if something accidentally modifies `df` later, we still have the correct reference numbers.

In [69]:
baseline = {
    'rows'           : df.shape[0],
    'columns'        : df.shape[1],
    'missing_values' : int(df.isnull().sum().sum()),
    'duplicate_rows' : int(df.duplicated().sum()),
    'TotalCharges_dtype': str(df['TotalCharges'].dtype),
}

print('=== BEFORE Cleaning ===')
for k, v in baseline.items():
    print(f'  {k:<25}: {v}')

=== BEFORE Cleaning ===
  rows                     : 7043
  columns                  : 21
  missing_values           : 0
  duplicate_rows           : 0
  TotalCharges_dtype       : object


---
## 3. Create the Working Copy

**Why copy?**  
`df.copy()` creates a completely independent DataFrame in memory.  
Any operation on `df_clean` will NOT affect `df` — the raw data remains a pristine reference throughout the session.  

Using a copy instead of modifying `df` in place is a data engineering best practice: if a cleaning step turns out to be wrong, you can always regenerate `df_clean` from `df` without re-reading the file.

In [70]:
df_clean = df.copy()

print('df_clean created — an independent copy of df.')
print(f'Shape: {df_clean.shape}')
print(f'Is df_clean the same object as df? {df_clean is df}')

df_clean created — an independent copy of df.
Shape: (7043, 21)
Is df_clean the same object as df? False


---
## 4. Drop `customerID`

**Why remove it?**  
`customerID` is a unique string identifier assigned to each customer record. It carries zero predictive information:  
- It has one unique value per row (7,043 unique values for 7,043 rows).
- It is not a measurement of any customer behaviour or attribute.
- Including it in a model would either be ignored or — in worst-case scenarios with certain algorithms — cause data leakage or artificial pattern-matching on arbitrary IDs.

**Why only drop from `df_clean` and not from `df`?**  
The raw DataFrame `df` is kept intact. `customerID` could be useful later for joining results back to a business database or for traceability/debugging.

`inplace=True` modifies `df_clean` directly without needing a reassignment.

In [71]:
print('customerID present in df       :', 'customerID' in df.columns)       # raw — untouched
print('customerID present in df_clean :', 'customerID' in df_clean.columns) # before drop

df_clean.drop(columns=['customerID'], inplace=True)

print('customerID present in df_clean :', 'customerID' in df_clean.columns) # after drop
print(f'df_clean shape after drop: {df_clean.shape}')

customerID present in df       : True
customerID present in df_clean : True
customerID present in df_clean : False
df_clean shape after drop: (7043, 20)


---
## 5. Handle Duplicate Rows

**Why check for duplicates?**  
Duplicate rows inflate counts, distort statistical summaries, and give a model repeated identical training examples — effectively giving that record more weight without justification.

Step 2 found **zero duplicates** in the raw data, but we re-check on `df_clean` (after dropping `customerID`) because two rows that differed only by ID would now appear identical and should be caught here.  

If any duplicates are found, we drop all copies except the first occurrence (`keep='first'`) and report the change.

In [72]:
n_dup = df_clean.duplicated().sum()
print(f'Duplicate rows in df_clean (post customerID drop): {n_dup}')

if n_dup > 0:
    print('\nDuplicate rows found — displaying them:')
    display(df_clean[df_clean.duplicated(keep=False)])

    rows_before = len(df_clean)
    df_clean.drop_duplicates(keep='first', inplace=True)
    rows_after  = len(df_clean)
    print(f'\nRemoved {rows_before - rows_after} duplicate rows.')
    print(f'Shape after deduplication: {df_clean.shape}')
else:
    print('No duplicates found — nothing to remove. ✓')

Duplicate rows in df_clean (post customerID drop): 22

Duplicate rows found — displaying them:


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
22,Male,0,No,No,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.15,20.15,Yes
100,Male,0,No,No,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.20,20.2,No
542,Female,0,No,No,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,19.55,19.55,No
646,Male,0,No,No,1,Yes,No,DSL,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,45.70,45.7,Yes
662,Male,0,No,No,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.05,20.05,No
690,Male,0,No,No,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Mailed check,20.45,20.45,No
964,Male,0,No,No,1,Yes,No,DSL,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,45.70,45.7,Yes
976,Male,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.90,69.9,Yes
1243,Male,0,No,No,1,Yes,No,DSL,No,No,No,No,No,No,Month-to-month,No,Electronic check,45.30,45.3,Yes
1338,Male,0,No,No,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.15,20.15,Yes



Removed 22 duplicate rows.
Shape after deduplication: (7021, 20)


---
## 6. Fix `TotalCharges` — Step A: Investigate Before Fixing

**What the audit found:**  
- `TotalCharges` was read as `object` (string) by pandas instead of `float64`.
- The reason: some entries contain a blank string `' '` (whitespace only) instead of a numeric value.
- `df.isnull()` reported 0 missing values in this column because blank strings are valid strings — not `NaN`.

**Before fixing anything**, we identify those rows precisely.  
We strip whitespace and check which entries become an empty string, then display the full context (including `tenure` and `MonthlyCharges`) to understand the pattern.

In [73]:
# Identify entries that are blank or whitespace-only
blank_mask = df_clean['TotalCharges'].astype(str).str.strip() == ''
n_blank    = blank_mask.sum()

print(f'Blank / whitespace-only entries in TotalCharges: {n_blank}')

if n_blank > 0:
    print('\nFull rows where TotalCharges is blank:')
    display(df_clean[blank_mask])

Blank / whitespace-only entries in TotalCharges: 11

Full rows where TotalCharges is blank:


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,No,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,,No
753,Male,0,No,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,,No
936,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,Yes,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,,No
1082,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,,No
1340,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,Yes,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,,No
3331,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,,No
3826,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,,No
4380,Female,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,,No
5218,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,,No
6670,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,Yes,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,,No


In [74]:
# Cross-check: are all blank TotalCharges rows also tenure == 0?
print('tenure values for blank-TotalCharges rows:')
print(df_clean.loc[blank_mask, 'tenure'].value_counts().to_string())

print('\nMonthlyCharges values for blank-TotalCharges rows:')
print(df_clean.loc[blank_mask, 'MonthlyCharges'].describe().to_string())

tenure values for blank-TotalCharges rows:
tenure
0    11

MonthlyCharges values for blank-TotalCharges rows:
count    11.000000
mean     41.418182
std      23.831484
min      19.700000
25%      20.125000
50%      25.750000
75%      58.975000
max      80.850000


### ✏️ Decision: How to treat these 11 rows

**Observation:**  
All rows where `TotalCharges` is blank have `tenure = 0`, meaning these customers joined the company but were billed for zero full months at the time the snapshot was taken.

**Options considered:**

| Option | Approach | Risk |
|---|---|---|
| A — Impute with `0` | Set `TotalCharges = 0` for tenure-0 customers | Consistent with reality (no complete billing period yet); mathematically sound |
| B — Impute with `MonthlyCharges` | Use each row's own `MonthlyCharges` as a proxy | Overestimates actual charges; not justified |
| C — Drop the rows | Remove 11 rows from the dataset | Loses 11 valid customer records; slightly reduces data; acceptable but unnecessary |
| D — Leave as `NaN` | Convert blank → `NaN`, do not impute | Most ML pipelines require no `NaN`; creates downstream complexity |

**Chosen: Option A — Impute with `0`**  
A customer with `tenure = 0` has completed zero billing cycles. Their total charges are genuinely `0.0` — the blank is a data entry artefact from the source system, not a true unknown value. Setting it to `0` is factually correct and loses no information.  
We do **not** drop any rows.

## 6. Fix `TotalCharges` — Step B: Convert and Impute

**Process:**
1. `pd.to_numeric(..., errors='coerce')` tries to parse every value as a float. Anything that fails (the blank strings) is silently replaced with `NaN`. This is how we turn invisible blank strings into proper missing-value markers.
2. `.fillna(0.0)` replaces those `NaN` values with `0.0` — our chosen imputation strategy for tenure-0 customers.
3. The result is reassigned back to `df_clean['TotalCharges']`, which now holds proper `float64` values.

We verify the conversion at the end by printing the new dtype and confirming zero remaining `NaN` values in this column.

In [75]:
print(f'TotalCharges dtype BEFORE: {df_clean["TotalCharges"].dtype}')
print(f'TotalCharges NaN count BEFORE: {df_clean["TotalCharges"].isnull().sum()}')

# Step 1: coerce — blank strings become NaN
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')

n_nan_after_coerce = df_clean['TotalCharges'].isnull().sum()
print(f'\nTotalCharges NaN count AFTER coerce (should match blank count): {n_nan_after_coerce}')

# Step 2: fill the NaNs with 0.0 (tenure-0 customers have not completed a billing cycle)
df_clean['TotalCharges'] = df_clean['TotalCharges'].fillna(0.0)

print(f'TotalCharges NaN count AFTER fillna(0): {df_clean["TotalCharges"].isnull().sum()}')
print(f'TotalCharges dtype AFTER: {df_clean["TotalCharges"].dtype}')

TotalCharges dtype BEFORE: object
TotalCharges NaN count BEFORE: 0

TotalCharges NaN count AFTER coerce (should match blank count): 11
TotalCharges NaN count AFTER fillna(0): 0
TotalCharges dtype AFTER: float64


In [76]:
# Sanity-check: verify the previously-blank rows now have TotalCharges == 0.0
print('Rows that previously had blank TotalCharges (should now show 0.0):')
display(df_clean.loc[blank_mask, ['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].head(15))

Rows that previously had blank TotalCharges (should now show 0.0):


,tenure,MonthlyCharges,TotalCharges,Churn
488,0,52.55,0.0,No
753,0,20.25,0.0,No
936,0,80.85,0.0,No
1082,0,25.75,0.0,No
1340,0,56.05,0.0,No
3331,0,19.85,0.0,No
3826,0,25.35,0.0,No
4380,0,20.00,0.0,No
5218,0,19.70,0.0,No
6670,0,73.35,0.0,No


---
## 7. Retain `SeniorCitizen` as Numerical (0 / 1)

**Decision: Keep as `int64` for now.**

**Why not convert to `'Yes'`/`'No'`?**  
- `SeniorCitizen` is already a clean binary indicator with no missing values and no inconsistent casing.
- Converting it to a string now would require encoding it back to a number before model training anyway (Step 6). We skip the unnecessary round-trip.
- Many ML algorithms treat `int64` binary columns natively without requiring explicit encoding.
- When we encode all categorical columns uniformly in Step 5 (Feature Engineering), we will decide on the final encoding strategy for this column at the same time.

We simply confirm its current state is correct.

In [77]:
print('SeniorCitizen dtype  :', df_clean['SeniorCitizen'].dtype)
print('SeniorCitizen unique :', sorted(df_clean['SeniorCitizen'].unique().tolist()))
print('SeniorCitizen nulls  :', df_clean['SeniorCitizen'].isnull().sum())
print('\nRetained as int64 (0/1). No change needed at this stage. ✓')

SeniorCitizen dtype  : int64
SeniorCitizen unique : [0, 1]
SeniorCitizen nulls  : 0

Retained as int64 (0/1). No change needed at this stage. ✓


---
## 8. Retain `No internet service` and `No phone service` Categories

**Decision: Keep as-is for now.**

**Why not collapse them into `'No'`?**  
- `'No internet service'` is genuinely informative: it tells us the customer does not have internet at all — a different state from a customer who has internet but chose not to add online security.
- Collapsing it prematurely could destroy signal that EDA (Step 4) might reveal to be predictive of churn.
- The right time to make this decision is in Step 5 (Feature Engineering), once we understand the distributions and correlations.

We document which columns contain these values without modifying them.

In [78]:
cols_no_internet_svc = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]
cols_no_phone_svc = ['MultipleLines']

print('Columns retaining "No internet service" as a category:')
for col in cols_no_internet_svc:
    vals = sorted(df_clean[col].unique().tolist())
    print(f'  {col:<25} → {vals}')

print('\nColumns retaining "No phone service" as a category:')
for col in cols_no_phone_svc:
    vals = sorted(df_clean[col].unique().tolist())
    print(f'  {col:<25} → {vals}')

print('\nNo changes made — retained intentionally for EDA and Feature Engineering. ✓')

Columns retaining "No internet service" as a category:
  OnlineSecurity            → ['No', 'No internet service', 'Yes']
  OnlineBackup              → ['No', 'No internet service', 'Yes']
  DeviceProtection          → ['No', 'No internet service', 'Yes']
  TechSupport               → ['No', 'No internet service', 'Yes']
  StreamingTV               → ['No', 'No internet service', 'Yes']
  StreamingMovies           → ['No', 'No internet service', 'Yes']

Columns retaining "No phone service" as a category:
  MultipleLines             → ['No', 'No phone service', 'Yes']

No changes made — retained intentionally for EDA and Feature Engineering. ✓


---
## 9. Post-Cleaning Integrity Checks

After applying all cleaning operations, we re-run every quality check from Step 2 on `df_clean`.  
The goal is to confirm that:
1. All intended fixes were applied correctly.
2. No new issues were accidentally introduced by the cleaning steps.

These checks mirror Step 2 exactly so results are directly comparable.

### 9.1 Shape

In [79]:
print(f'df_clean shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns')

df_clean shape: 7,021 rows × 20 columns


### 9.2 Data Types

In [80]:
display(df_clean.dtypes.to_frame(name='dtype'))

,dtype
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
PhoneService,object
MultipleLines,object
InternetService,object
OnlineSecurity,object
OnlineBackup,object


### 9.3 Missing Values

In [81]:
total_missing = df_clean.isnull().sum().sum()
print(f'Total missing values (NaN/None) in df_clean: {total_missing}')

per_col = df_clean.isnull().sum()
missing_cols = per_col[per_col > 0]
if missing_cols.empty:
    print('No missing values remaining. ✓')
else:
    print('Columns still missing values:')
    display(missing_cols.to_frame(name='Missing Count'))

Total missing values (NaN/None) in df_clean: 0
No missing values remaining. ✓


### 9.4 Duplicate Rows

In [82]:
n_dup_after = df_clean.duplicated().sum()
print(f'Duplicate rows in df_clean: {n_dup_after}')
if n_dup_after == 0:
    print('No duplicates. ✓')

Duplicate rows in df_clean: 0
No duplicates. ✓


### 9.5 Negative and Infinite Values in Numeric Columns

In [83]:
num_cols_clean = df_clean.select_dtypes(include=[np.number]).columns.tolist()
print(f'Numeric columns now: {num_cols_clean}\n')

issues = []
for col in num_cols_clean:
    s = df_clean[col]
    issues.append({
        'Column'        : col,
        'Negative Count': int((s < 0).sum()),
        'Infinite Count': int(np.isinf(s).sum()),
        'NaN Count'     : int(s.isnull().sum()),
        'Min'           : round(s.min(), 4),
        'Max'           : round(s.max(), 4)
    })

issues_df = pd.DataFrame(issues).set_index('Column')
display(issues_df)

flag = issues_df[(issues_df['Negative Count'] > 0) | (issues_df['Infinite Count'] > 0) | (issues_df['NaN Count'] > 0)]
if flag.empty:
    print('\nNo negative, infinite, or NaN values in numeric columns. ✓')
else:
    print('\n⚠️  Issues still present:')
    display(flag)

Numeric columns now: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']



,Negative Count,Infinite Count,NaN Count,Min,Max
Column,,,,,
SeniorCitizen,0,0,0,0.00,1.00
tenure,0,0,0,0.00,72.00
MonthlyCharges,0,0,0,18.25,118.75
TotalCharges,0,0,0,0.00,8684.80



No negative, infinite, or NaN values in numeric columns. ✓


### 9.6 Numerical Summary Statistics

In [84]:
display(df_clean[num_cols_clean].describe().T)

,count,mean,std,min,25%,50%,75%,max
SeniorCitizen,7021.0,0.162512,0.368947,0.00,0.00,0.00,0.0,1.00
tenure,7021.0,32.469449,24.534965,0.00,9.00,29.00,55.0,72.00
MonthlyCharges,7021.0,64.851894,30.069001,18.25,35.75,70.40,89.9,118.75
TotalCharges,7021.0,2286.765026,2266.855057,0.00,403.35,1400.55,3801.7,8684.80


### 9.7 Categorical Columns — Unique Values

In [85]:
cat_cols_clean = df_clean.select_dtypes(include='object').columns.tolist()
print(f'Categorical (object) columns: {len(cat_cols_clean)}\n')

for col in cat_cols_clean:
    vals = sorted(df_clean[col].dropna().unique().tolist())
    print(f'  {col:<30} ({df_clean[col].nunique()} unique) → {vals}')

Categorical (object) columns: 16

  gender                         (2 unique) → ['Female', 'Male']
  Partner                        (2 unique) → ['No', 'Yes']
  Dependents                     (2 unique) → ['No', 'Yes']
  PhoneService                   (2 unique) → ['No', 'Yes']
  MultipleLines                  (3 unique) → ['No', 'No phone service', 'Yes']
  InternetService                (3 unique) → ['DSL', 'Fiber optic', 'No']
  OnlineSecurity                 (3 unique) → ['No', 'No internet service', 'Yes']
  OnlineBackup                   (3 unique) → ['No', 'No internet service', 'Yes']
  DeviceProtection               (3 unique) → ['No', 'No internet service', 'Yes']
  TechSupport                    (3 unique) → ['No', 'No internet service', 'Yes']
  StreamingTV                    (3 unique) → ['No', 'No internet service', 'Yes']
  StreamingMovies                (3 unique) → ['No', 'No internet service', 'Yes']
  Contract                       (3 unique) → ['Month-to-month', 'On

---
## 10. Before-vs-After Summary

A side-by-side comparison of the raw DataFrame (`df`) and the cleaned DataFrame (`df_clean`) across the key quality dimensions.

This provides a concise audit trail that can be referenced in the project report.

In [86]:
after = {
    'rows'              : df_clean.shape[0],
    'columns'           : df_clean.shape[1],
    'missing_values'    : int(df_clean.isnull().sum().sum()),
    'duplicate_rows'    : int(df_clean.duplicated().sum()),
    'TotalCharges_dtype': str(df_clean['TotalCharges'].dtype),
}

comparison = pd.DataFrame({
    'Before (raw)' : baseline,
    'After (clean)': after
})

# Add a change column
comparison['Changed?'] = comparison.apply(
    lambda row: '✅ Yes' if str(row['Before (raw)']) != str(row['After (clean)']) else '— No',
    axis=1
)

print('=== Before vs After Cleaning ===')
display(comparison)

=== Before vs After Cleaning ===


,Before (raw),After (clean),Changed?
rows,7043,7021,✅ Yes
columns,21,20,✅ Yes
missing_values,0,0,— No
duplicate_rows,0,0,— No
TotalCharges_dtype,object,float64,✅ Yes


### dtype changes specifically

In [87]:
dtype_before = df.dtypes.rename('Before')
dtype_after  = df_clean.dtypes.rename('After')

# Align on column names (df has customerID, df_clean does not)
dtype_comparison = pd.concat([dtype_before, dtype_after], axis=1)
dtype_comparison['Changed?'] = dtype_comparison.apply(
    lambda row: '✅ Yes' if str(row['Before']) != str(row['After']) else '—',
    axis=1
)

display(dtype_comparison)

,Before,After,Changed?
customerID,object,NaN,✅ Yes
gender,object,object,—
SeniorCitizen,int64,int64,—
Partner,object,object,—
Dependents,object,object,—
tenure,int64,int64,—
PhoneService,object,object,—
MultipleLines,object,object,—
InternetService,object,object,—
OnlineSecurity,object,object,—


---
## 11. Save the Cleaned Dataset

We save `df_clean` to a **separate file** — `data/telco_churn_cleaned.csv`.  
The raw file `WA_Fn-UseC_-Telco-Customer-Churn.csv` is never touched.

`index=False` prevents pandas from writing the integer row index as an extra column in the CSV — it is not a meaningful data column.

In [88]:
df_clean.to_csv(CLEAN_PATH, index=False)

print(f'Cleaned dataset saved to: {CLEAN_PATH}')
print(f'File size: {os.path.getsize(CLEAN_PATH) / 1024:.1f} KB')

# Read back one row to confirm the file was written correctly
verify = pd.read_csv(CLEAN_PATH, nrows=3)
print('\nFirst 3 rows of saved file:')
display(verify)

Cleaned dataset saved to: c:\Users\Subham\OneDrive\Documents\Desktop\Customer_Churn_Project\data\telco_churn_cleaned.csv
File size: 876.9 KB

First 3 rows of saved file:


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


---

# 🧹 Data Cleaning Summary

---

## What Was Cleaned

### `TotalCharges` — Type Fix and Imputation
- **Issue:** The column was stored as `object` (string) because 11 entries contained a blank/whitespace string `' '` instead of a numeric value. `pandas` could not infer the column as numeric due to these entries.
- **Fix applied:**
  1. Used `pd.to_numeric(..., errors='coerce')` to convert the column — blank strings became `NaN`.
  2. Used `.fillna(0.0)` to replace those 11 `NaN` values with `0.0`.
- **Justification:** All 11 affected rows had `tenure = 0`. A customer with zero completed billing periods has an actual total charge of `0.0`. The blank was a source-system artefact, not a genuinely unknown value. Imputing with `0` is factually correct and avoids dropping valid records.
- **Result:** `TotalCharges` dtype changed from `object` → `float64`. Zero missing values remain.

---

## What Was Removed

### `customerID` — Dropped from `df_clean`
- **Issue:** Pure row identifier — 7,043 unique values for 7,043 rows. Carries no predictive signal.
- **Fix applied:** `df_clean.drop(columns=['customerID'], inplace=True)`
- **Justification:** Identifier columns can silently harm model training if included (risk of spurious pattern matching). Their only legitimate use is as a lookup key for business systems, which is a post-modelling concern.
- **Important:** `customerID` was dropped **only from `df_clean`**. The original raw DataFrame `df` still contains it and the raw CSV is unchanged.

### Duplicate rows
- **Result from Step 2 audit:** 0 duplicates in the raw data.
- **Re-checked** in this notebook after dropping `customerID` — still 0 duplicates.
- **Action taken:** None (no duplicates to remove).

---

## What Was Intentionally Retained

### `SeniorCitizen` — Kept as `int64` (0/1)
- **Why not converted to `'Yes'`/`'No'`?** It is already a clean binary numeric column. Converting to string now would require encoding it back to numeric before modelling — a pointless round-trip. The encoding strategy for all categorical and binary columns will be decided uniformly in Step 5.

### `'No internet service'` and `'No phone service'` — Kept as separate categories
- **Affected columns:** `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`, `MultipleLines`.
- **Why not collapsed to `'No'`?** These values carry distinct meaning — a customer without internet is in a fundamentally different situation from one who has internet but declined a specific add-on. Collapsing before EDA could destroy predictive signal. This decision will be revisited in Step 5 (Feature Engineering) with EDA evidence to support or refute it.

### Class Imbalance in `Churn`
- The target variable remains ~73% `No` / ~27% `Yes`. This imbalance is a **modelling concern**, not a data cleaning issue. It will be handled in Step 6 using techniques such as stratified splitting, class-weight parameters, or oversampling (SMOTE).

---

## Files
| File | Status |
|---|---|
| `data/WA_Fn-UseC_-Telco-Customer-Churn.csv` | ✅ Unchanged — raw file preserved |
| `data/telco_churn_cleaned.csv` | ✅ Created — clean version for downstream steps |

---

**Next step →** Step 4: Exploratory Data Analysis — visualise distributions, correlations, and churn patterns using `df_clean` (or load from `telco_churn_cleaned.csv`).